# AM322 - Bayesian optimisation and HMM

The initial 50-trial study uses the five available training trips and logistic regression as a provisional model. GroupKFold keeps each trip wholly in a single validation fold. Optuna minimises candidate-level log loss because the decoder consumes the classifier's probabilities. Candidate macro F1 is also recorded at a 0.5 threshold.

The training probabilities are generated by models that did not fit the scored trip. Those same folds are used to select hyperparameters, so these sample results are exploratory. Once the team hands off the selected estimator and full tables, use the held-out student-test table to assess the tuned classifier and decoder.

## Reproduce the sample

From the repository root, run the following commands (replace `<raw-train-dir>` with the directory containing the raw training trip folders).

```text
python -m src.tuning.bayes_opt --train data/processed/sample_candidates.parquet --output results/hpo/sample
python -m src.hmm.evaluate_hmm --candidates results/hpo/sample/oof_probabilities.parquet --raw-root "<raw-train-dir>" --probability-column probability --output results/hmm/classifier_sample_comparison.csv
```

## Run the team-selected model

Use AM321's selection CSV after generating the full candidate tables. This run reuses the team estimator, seed, trusted training rows, and training-row cap.

```text
python -m src.tuning.bayes_opt --selected-model results/models/selected_model.csv --student-test data/processed/student_test_candidates.parquet --artifacts-dir artifacts
python -m src.hmm.evaluate_hmm --candidates results/hpo/student_test_probabilities.parquet --raw-root "<raw-student-test-dir>" --probability-column probability --output results/hmm/student_test_comparison.csv
```


In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent

display(pd.read_csv(root / 'results/hpo/sample/comparison.csv'))
display(pd.read_csv(root / 'results/hmm/classifier_sample_comparison.csv', dtype={'trip_id': str}))